# Activations & Losses

激活函数给网络"非线性"能力，损失函数定义"好"与"差"。本课对比主流激活与损失，并证明一个关键事实：**没有非线性的多层网络等于一层**。


## 0. 环境配置与导入


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib

matplotlib.rcParams["font.sans-serif"] = ["PingFang SC", "Hiragino Sans GB", "Arial Unicode MS", "Microsoft YaHei", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

print("PyTorch version:", torch.__version__)
torch.manual_seed(42)


## 1. 激活函数全家桶


| 激活 | 公式 | 导数 | 特点 |
|------|------|------|------|
| sigmoid | $\sigma(z)=\frac{1}{1+e^{-z}}$ | $\sigma(1-\sigma)$ | 输出 (0,1)，两端饱和 |
| tanh | $\tanh z$ | $1-\tanh^2 z$ | 输出 (-1,1)，零中心 |
| ReLU | $\max(0,z)$ | $\mathbb{1}[z>0]$ | 稀疏、无饱和，负区死 |
| LeakyReLU | $\max(\alpha z, z)$ | $\alpha$ 或 1 | 解决 ReLU 死亡 |
| ELU | $z\ (z>0),\ \alpha(e^z-1)$ | 分段 | 负区平滑 |


In [ ]:
def act_fns():
    return {
        'sigmoid': (lambda z: 1/(1+np.exp(-z)),
                    lambda z: (1/(1+np.exp(-z)))*(1-1/(1+np.exp(-z)))),
        'tanh':    (lambda z: np.tanh(z), lambda z: 1-np.tanh(z)**2),
        'ReLU':    (lambda z: np.maximum(0, z), lambda z: (z > 0).astype(float)),
        'LeakyReLU': (lambda z: np.where(z > 0, z, 0.1*z), lambda z: np.where(z > 0, 1.0, 0.1)),
        'ELU':     (lambda z: np.where(z > 0, z, np.exp(z)-1), lambda z: np.where(z > 0, 1.0, np.exp(z))),
    }
z = np.linspace(-4, 4, 300)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, (f, df) in act_fns().items():
    axes[0].plot(z, f(z), label=name)
    axes[1].plot(z, df(z), label=name)
axes[0].set_title('激活函数'); axes[1].set_title('导数（梯度可流动区域）')
for ax in axes: ax.set_xlabel('z'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()


## 2. 为什么需要非线性：线性堆叠 = 单层


两层线性层 $W_2(W_1 x + b_1) + b_2 = (W_1W_2)x + (b_1W_2 + b_2)$——还是一个线性函数！

**证明**：$n$ 层线性层总可合并为单层，表达能力与单层相同。非线性激活（ReLU/tanh…）是"深度"能工作的必要条件。


In [ ]:
rng = np.random.default_rng(0)
W1 = rng.standard_normal((3, 4)); b1 = rng.standard_normal(4)
W2 = rng.standard_normal((4, 2)); b2 = rng.standard_normal(2)
x = rng.standard_normal(3)

two_layer = (x @ W1 + b1) @ W2 + b2
W_eff = W1 @ W2;  b_eff = b1 @ W2 + b2
one_layer = x @ W_eff + b_eff

print(f"两层线性与单层最大差 = {np.abs(two_layer - one_layer).max():.2e}")
print("→ 数学上完全等价：没有非线性就没有深度")


## 3. 损失函数：回归与分类


| 场景 | 损失 | 公式 | 分布假设 |
|------|------|------|----------|
| 回归 | MSE | $(y-\hat y)^2$ | 高斯噪声 |
| 回归 | MAE | $|y-\hat y|$ | 拉普拉斯噪声 |
| 回归 | Huber | 分段二次/线性 | 抗离群点 |
| 二分类 | BCE | $-y\log\hat y -(1-y)\log(1-\hat y)$ | 伯努利 |
| 多分类 | CE | $-\log \hat p_y$ | 类别分布 |

（对应概率 05 课"损失 = 负对数似然"对照表。）


In [ ]:
def mse(d): return d**2
def mae(d): return np.abs(d)
def huber(d, delta=1.0):
    a = np.abs(d)
    return np.where(a <= delta, 0.5*a**2, delta*(a - 0.5*delta))

d = np.linspace(-4, 4, 400)
plt.figure(figsize=(8, 4.5))
plt.plot(d, mse(d), label='MSE')
plt.plot(d, mae(d), label='MAE')
plt.plot(d, huber(d), label='Huber (δ=1)')
plt.xlabel('预测误差 y − ŷ'); plt.ylabel('损失')
plt.title('回归损失对比：离群点敏感性')
plt.legend(); plt.grid(alpha=0.3)
print("误差=3 时: MSE =", mse(3), " MAE =", mae(3), " Huber =", huber(3))
print("→ MSE 对离群点惩罚是平方级，MAE 线性，Huber 折中")


## 4. 损失的可微性：为什么 0-1 错误率不能直接优化


分类的终极指标是 0-1 错误率（$\mathbb{1}[\hat y \ne y]$），但它处处不可微、梯度为零——没法梯度下降。交叉熵是它的"可微替身"：形状接近 0-1 损失，但有平滑的梯度信号。


In [ ]:
# 二分类：交叉熵 vs 0-1 损失的代理对比
z = np.linspace(-6, 6, 300)                       # logit，y=1
ce = np.log(1 + np.exp(-z))                       # BCE(σ(z), 1)
zero_one = (z < 0).astype(float)

plt.figure(figsize=(8, 4.5))
plt.plot(z, zero_one, 'k--', label='0-1 错误率（不可微）')
plt.plot(z, ce, 'r-', label='BCE（平滑代理）')
plt.xlabel('logit z（y=1）'); plt.ylabel('损失')
plt.title('交叉熵是 0-1 损失的平滑代理')
plt.legend(); plt.grid(alpha=0.3)
print("BCE 在 z→-∞ 时 →", ce[-1], "，0-1 损失 → 1；但 BCE 梯度处处存在")


## 5. 选择指南


- **回归**：噪声轻 → MSE；有离群点 → Huber；对尺度不敏感 → MAE
- **分类**：输出层用 softmax（多类）或 sigmoid（二类），损失用 CE/BCE——**不要**用 MSE 配分类输出（梯度弱且非凸性差）
- **数值稳定**：`F.cross_entropy` 把 softmax 与 log 合并计算（log-sum-exp 技巧，见 calculus 08），手写时勿分开


## 课后练习


1. **证明**：$n$ 层线性网络的输出仍是输入的线性函数（数学归纳法）。
2. **画图**：把 LeakyReLU 的 $\alpha$ 从 0.01 调到 0.5，观察导数形状变化。
3. **数值验证**：用中心差分验证 BCE 对 logit 的导数 = $\sigma(z)-y$。
4. **对比实验**：同一模型分别用 MSE 与 BCE 训练二分类，比较收敛速度。
5. **思考**：为什么 softmax + MSE 训练分类会"梯度太小"？（提示：饱和区的导数）
